# 🧪 Lab 3 — Praktikum Machine Learning Klasik> Pendamping materi Bab 3 — bab terpenting di buku ini. Prinsip: **setiap> algoritma dibangun dari nol dulu, baru dipakai lewat library**. Semua kode> di lab ini numpy murni (tanpa sklearn) supaya isi kap mesinnya terlihat.| Bagian | Topik | Waktu (menit) ||---|---|---|| 1 | Regresi linear dari nol + fitur polinomial | 25 || 2 | Logistic regression dari nol | 20 || 3 | Metrik evaluasi & data imbalanced | 25 || 4 | Bias–variance & regularisasi (ridge) | 20 || 5 | Decision tree & random forest mini | 25 || 6 | K-means dari nol | 15 |Setiap latihan punya cell **✅ Cek** — jalankan, kalau ✅ berarti pemahamanmu benar.

## Bagian 1 — Regresi Linear dari NolModel `ŷ = Xw + b`, loss MSE, dioptimasi gradient descent. Ini persisgradient descent yang kamu visualisasikan di Bab 2 — kini dengan data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 80
x = rng.uniform(0, 2, size=n)                      # fitur
y = 4 + 3 * x + rng.normal(0, 0.5, size=n)         # target: y = 4 + 3x + noise
X = x.reshape(-1, 1)                               # bentuk (n_sampel, n_fitur)

class LinearRegressorGD:
    def __init__(self, lr=0.1, n_iter=1500):
        self.lr = lr
        self.n_iter = n_iter
        self.loss_history = []

    def fit(self, X, y):
        m, nf = X.shape
        self.w = np.zeros(nf)
        self.b = 0.0
        self.loss_history = []
        for _ in range(self.n_iter):
            pred = X @ self.w + self.b
            error = pred - y
            self.loss_history.append(float(np.mean(error ** 2)))
            self.w -= self.lr * (2 / m) * (X.T @ error)
            self.b -= self.lr * (2 / m) * np.sum(error)
        return self

    def predict(self, X):
        return np.asarray(X) @ self.w + self.b

model = LinearRegressorGD(lr=0.1).fit(X, y)
print(f'w = {model.w[0]:.3f}  (seharusnya ~3.0)')
print(f'b = {model.b:.3f}  (seharusnya ~4.0)')
print(f'MSE akhir = {model.loss_history[-1]:.4f}')


In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.scatter(x, y, alpha=0.5, label='data')
xs = np.linspace(0, 2, 50)
plt.plot(xs, model.predict(xs.reshape(-1, 1)), 'r-', lw=2, label='model GD')
plt.plot(xs, 4 + 3 * xs, 'g--', label='y = 4+3x (asli)')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.title('Fit model')

plt.subplot(1, 2, 2)
plt.plot(model.loss_history)
plt.xlabel('iterasi'); plt.ylabel('MSE'); plt.title('Kurva loss — harus menurun')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Observasi:** loss turun cepat di awal lalu melandai — pola khas gradientdescent. Coba ganti `lr=1.0`: loss meledak (divergen). `lr=0.001`: lambat sekali.Learning rate adalah hyperparameter paling pertama yang dicek saat training aneh.

### 🎯 Latihan 1.1 — Fitur PolinomialModel linear hanya bisa garis lurus. Trik klasik: ubah fitur `x` menjadi`[x, x², x³, …]` — model tetap "linear terhadap parameter" tapi kurvanyabisa melengkung. Ini juga pintu masuk memahami overfitting (Bagian 4).

In [ ]:
def poly_features(x, degree):
    """Return matriks (n, degree) dengan kolom x^1, x^2, ..., x^degree.
    Input x: array 1D sepanjang n.
    """
    # TODO: bangun matriksnya tanpa loop per elemen (hint: broadcasting)
    raise NotImplementedError

# --- uji sendiri ---
# print(poly_features(np.array([2.0]), 3))   # [[2., 4., 8.]]
# print(poly_features(np.array([1., 2., 3.]), 2).shape)   # (3, 2)


In [ ]:
# ✅ Cek latihan 1.1
P1 = poly_features(np.array([2.0]), 3)
assert np.allclose(P1, [[2.0, 4.0, 8.0]]), f'nilai salah: {P1}'
P2 = poly_features(np.array([1., 2., 3.]), 2)
assert P2.shape == (3, 2)
assert np.allclose(P2[:, 1], [1., 4., 9.]), 'kolom kedua harus x^2'
P7 = poly_features(np.array([1.5]), 7)
assert np.allclose(P7, [[1.5 ** k for k in range(1, 8)]])
print('✅ Latihan 1.1 benar!')


In [ ]:
# Demo: pakai poly_features untuk fit kurva non-linear
xs = np.linspace(-3, 3, 60)
ys = np.sin(xs) * 2 + 0.5

P = poly_features(xs, 5)
w_poly = np.linalg.lstsq(P, ys, rcond=None)[0]      # least squares closed-form
print('koefisien poly deg-5:', w_poly.round(3))

plt.figure(figsize=(6, 4))
plt.scatter(xs, ys, s=12, alpha=0.6, label='data (sin)')
plt.plot(xs, poly_features(xs, 5) @ w_poly, 'r-', label='fit poly deg-5')
plt.legend(); plt.title('Linear + fitur polinomial bisa melengkung'); plt.show()


## Bagian 2 — Logistic Regression dari NolKlasifikasi biner: sama-sama linear (`z = Xw + b`), lalu sigmoid mengubah`z` jadi probabilitas 0–1. Loss-nya binary cross-entropy, bukan MSE.

In [ ]:
def sigmoid(z):
    """Versi stabil: cabang z>=0 dan z<0 supaya exp tidak overflow."""
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

z = np.linspace(-6, 6, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color='gray', ls=':'); plt.axvline(0, color='gray', ls=':')
plt.title('Sigmoid: z=0 -> p=0.5 (decision boundary)')
plt.xlabel('z'); plt.ylabel('p'); plt.show()


In [ ]:
# Dataset klasifikasi 2 fitur: kelas ditentukan sisi garis + noise
rng = np.random.default_rng(0)
n = 200
Xc = rng.normal(0, 1, size=(n, 2))
yc = (Xc[:, 0] + 0.8 * Xc[:, 1] + 0.4 * rng.normal(0, 1, n) > 0).astype(int)
print(f'distribusi kelas: {np.bincount(yc)}')

class LogisticGD:
    """Logistic regression full-batch GD. Gradien: X.T(p - y)/m."""
    def __init__(self, lr=0.5, n_iter=800):
        self.lr = lr
        self.n_iter = n_iter

    def fit(self, X, y):
        m = len(y)
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        for _ in range(self.n_iter):
            p = sigmoid(X @ self.w + self.b)
            self.w -= self.lr * (X.T @ (p - y)) / m
            self.b -= self.lr * np.sum(p - y) / m
        return self

    def predict_proba(self, X):
        return sigmoid(np.asarray(X) @ self.w + self.b)

clf = LogisticGD().fit(Xc, yc)
acc = float(((clf.predict_proba(Xc) >= 0.5).astype(int) == yc).mean())
print(f'w = {clf.w.round(3)}, b = {clf.b:.3f}, accuracy train = {acc:.3f}')
print('-> w mengikuti arah pemisah asli [1, 0.8] — model menemukannya sendiri.')


### 🎯 Latihan 2.1 — Forward Pass LogisticBangun fungsi prediksi probabilitas dari nol — tulang punggung classifier.

In [ ]:
def logistic_predict(X, w, b):
    """Return probabilitas kelas 1: sigmoid(X @ w + b).
    X: (n, d), w: (d,), b: float -> output bentuk (n,).
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# Xt = np.array([[0.0, 0.0], [10.0, 10.0]])
# wt = np.array([1.0, 1.0])
# print(logistic_predict(Xt, wt, 0.0))    # [0.5, ~1.0]


In [ ]:
# ✅ Cek latihan 2.1
Xt = np.array([[0.0, 0.0], [10.0, 10.0], [-10.0, -10.0]])
out = logistic_predict(Xt, np.array([1.0, 1.0]), 0.0)
assert out.shape == (3,)
assert abs(out[0] - 0.5) < 1e-9, 'z=0 harus p=0.5'
assert out[1] > 0.9999 and out[2] < 0.0001, 'z ekstrem harus mendekati 1/0'
out2 = logistic_predict(np.array([[1.0]]), np.array([2.0]), -1.0)
assert abs(out2[0] - 1 / (1 + np.exp(-1.0))) < 1e-9, 'z=2-1=1 -> sigma(1)=1/(1+e^-1)'
print('✅ Latihan 2.1 benar!')


## Bagian 3 — Metrik Evaluasi & Data Imbalanced> Bagian terpenting di lab ini. Accuracy saja bisa MENIPU.

In [ ]:
# Data kecil yang bisa dihitung manual
y_true = np.array([0, 0, 1, 1, 0, 1, 0, 0, 1, 1])   # 40% positif
y_pred = np.array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1])

TP = int(((y_true == 1) & (y_pred == 1)).sum())
TN = int(((y_true == 0) & (y_pred == 0)).sum())
FP = int(((y_true == 0) & (y_pred == 1)).sum())
FN = int(((y_true == 1) & (y_pred == 0)).sum())
print(f'TP={TP}  TN={TN}  FP={FP}  FN={FN}')
print(f'accuracy        = (TP+TN)/total = {(TP + TN) / 10:.2f}')
print(f'precision       = TP/(TP+FP)    = {TP / (TP + FP):.2f}')
print(f'recall          = TP/(TP+FN)    = {TP / (TP + FN):.2f}')
print(f'F1              = 2PR/(P+R)     = {2 * (TP / 5) * (TP / 5) / (TP / 5 + TP / 5):.2f}')


In [ ]:
# Demo: model BODOH pada data imbalanced — 90% negatif
rng = np.random.default_rng(1)
n = 1000
y_imb = (rng.random(n) < 0.1).astype(int)      # 10% positif
y_dumb = np.zeros(n, dtype=int)                # selalu tebak 'tidak'

acc_dumb = float((y_dumb == y_imb).mean())
rec_dumb = float(((y_imb == 1) & (y_dumb == 1)).sum() / max(1, (y_imb == 1).sum()))
print(f'accuracy model bodoh : {acc_dumb:.3f}  <- TINGGI!')
print(f'recall model bodoh   : {rec_dumb:.3f}  <- NOL: tidak menemukan satu pun positif')
print('-> Accuracy 90% tapi model TIDAK berguna. Selalu cek baseline bodoh dulu!')


### 🎯 Latihan 3.1 — Confusion Matrix dari Nol

In [ ]:
def confusion_counts(y_true, y_pred):
    """Return tuple (TP, TN, FP, FN). y hanya berisi 0/1."""
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# print(confusion_counts(y_true, y_pred))   # (4, 4, 1, 1)


In [ ]:
# ✅ Cek latihan 3.1
assert confusion_counts(y_true, y_pred) == (4, 4, 1, 1)
assert confusion_counts([0, 1, 1], [0, 0, 0]) == (0, 1, 0, 2)
assert confusion_counts([1, 1], [1, 1]) == (2, 0, 0, 0)
print('✅ Latihan 3.1 benar!')


### 🎯 Latihan 3.2 — Precision & Recall dengan GuardEdge case penting: kalau model **tidak pernah** memprediksi positif,precision = 0/(0) → harus di-guard jadi 0.0 (konvensi sklearn), bukan error.

In [ ]:
def precision_recall(y_true, y_pred):
    """Return (precision, recall).
    Guard: TP+FP == 0 -> precision 0.0 ; TP+FN == 0 -> recall 0.0.
    """
    # TODO: pakai confusion_counts dari latihan 3.1
    raise NotImplementedError

# --- uji sendiri ---
# print(precision_recall(y_true, y_pred))          # (0.8, 0.8)
# print(precision_recall([0, 1], [0, 0]))          # (0.0, 0.0) — bukan error!


In [ ]:
# ✅ Cek latihan 3.2
p, r = precision_recall(y_true, y_pred)
assert abs(p - 0.8) < 1e-9 and abs(r - 0.8) < 1e-9
assert precision_recall([0, 1, 1], [0, 0, 0]) == (0.0, 0.0), 'guard pembagian nol'
assert precision_recall([1, 0], [1, 1]) == (0.5, 1.0), 'precision 1/2, recall 1/1'
print('✅ Latihan 3.2 benar!')


In [ ]:
# Demo: threshold mengubah precision & recall (pakai model Bagian 2)
p_proba = clf.predict_proba(Xc)
print(f"{'th':>5} | {'precision':>9} | {'recall':>7}")
print('-' * 30)
for th in [0.2, 0.35, 0.5, 0.65, 0.8]:
    pred = (p_proba >= th).astype(int)
    tp = int(((yc == 1) & (pred == 1)).sum())
    fp = int(((yc == 0) & (pred == 1)).sum())
    fn = int(((yc == 1) & (pred == 0)).sum())
    pr = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    print(f'{th:>5.2f} | {pr:>9.3f} | {rc:>7.3f}')
print('-> threshold turun: recall naik, precision turun. Pilih sesuai BIAYA kesalahan.')


## Bagian 4 — Bias–Variance: Overfitting yang KelihatanRegresi polinomial derajat tinggi pada data sedikit = mesin overfitting.Kita buktikan dengan kurva train vs validation.

In [ ]:
rng = np.random.default_rng(7)
xs = np.sort(rng.uniform(-1, 1, size=30))
ys = np.sin(2 * np.pi * xs) * 0.8 + rng.normal(0, 0.15, 30)   # pola + noise

X_tr, y_tr = xs[:20], ys[:20]      # train
X_va, y_va = xs[20:], ys[20:]      # validation

derajat_list = [1, 3, 5, 9, 15, 25]
train_mse, val_mse = [], []

for d in derajat_list:
    Ptr = poly_features(X_tr, d)
    Pva = poly_features(X_va, d)
    w = np.linalg.lstsq(Ptr, y_tr, rcond=None)[0]
    train_mse.append(float(np.mean((Ptr @ w - y_tr) ** 2)))
    val_mse.append(float(np.mean((Pva @ w - y_va) ** 2)))

plt.figure(figsize=(7, 4))
plt.plot(derajat_list, train_mse, 'o-', label='train MSE')
plt.plot(derajat_list, val_mse, 's-', label='val MSE')
plt.yscale('log')
plt.xlabel('derajat polinomial'); plt.ylabel('MSE (log)')
plt.title('Derajat naik: train terus turun, val berbalik naik = OVERFIT')
plt.legend(); plt.grid(alpha=0.3); plt.show()

best = derajat_list[int(np.argmin(val_mse))]
print(f'derajat terbaik menurut validation: {best}')
print('-> model dipilih di VALIDATION, bukan train. Ini inti Bab 3!')


**Observasi kunci:** train MSE **monoton turun** saat derajat naik (model makinbefas dengan data train), tapi val MSE turun lalu **meledak**. Gap besartrain–val = overfitting. Gap besar di kedua tinggi = underfitting.

### 🎯 Latihan 4.1 — Ridge Regression (Regularisasi L2)Ridge menghukum bobot besar: `min ||Xw - y||² + λ||w||²`. Solusi closed-form:`w = (XᵀX + λI)⁻¹ Xᵀy`. λ besar → bobot menyusut → model lebih sederhana.

In [ ]:
def ridge_fit(X, y, lam):
    """Closed-form ridge. X: (n, d) TANPA kolom intercept, y: (n,).
    Return w: (d,).
    Hint: np.linalg.solve(X.T @ X + lam * I, X.T @ y)
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# P = poly_features(xs, 25)
# w0 = ridge_fit(P, ys, 0.0)
# w1 = ridge_fit(P, ys, 1.0)
# print(np.linalg.norm(w0), np.linalg.norm(w1))   # norm w1 harus lebih kecil


In [ ]:
# ✅ Cek latihan 4.1
P = poly_features(xs, 8)
w_ridge0 = ridge_fit(P, ys, 0.0)
w_lstsq = np.linalg.lstsq(P, ys, rcond=None)[0]
assert np.allclose(w_ridge0, w_lstsq, atol=1e-8), 'lam=0 harus sama dengan OLS'
norms = [np.linalg.norm(ridge_fit(P, ys, lam)) for lam in [0.0, 0.1, 1.0, 10.0, 100.0]]
assert all(norms[i] >= norms[i + 1] - 1e-10 for i in range(len(norms) - 1)), \
    'lambda naik -> norm bobot menyusut (monoton)'
print('✅ Latihan 4.1 benar! Regularisasi menyusutkan bobot seperti seharusnya.')


In [ ]:
# Demo: ridge menyelamatkan polinomial derajat tinggi
P25_tr = poly_features(X_tr, 25)
P25_va = poly_features(X_va, 25)

for lam in [0.0, 1e-4, 1e-2, 1.0]:
    w = ridge_fit(P25_tr, y_tr, lam)
    m_tr = float(np.mean((P25_tr @ w - y_tr) ** 2))
    m_va = float(np.mean((P25_va @ w - y_va) ** 2))
    print(f'lam={lam:>6}: train MSE={m_tr:.4f}  val MSE={m_va:.4f}  |w|={np.linalg.norm(w):8.2f}')
print('-> lambda pas memperkecil val MSE tanpa membuat train MBLEDAH.')


## Bagian 5 — Decision Tree & Random Forest MiniTree memutuskan seperti flowchart: pilih split yang membuat kelompokse-murni mungkin, diukur dengan **Gini impurity**. Kita bangun versi mini(cukup untuk memahami mekanisme — produksi tetap pakai sklearn/XGBoost).

### 🎯 Latihan 5.1 — Gini Impurity`Gini = 1 − Σ pₖ²` untuk proporsi kelas pₖ dalam satu kelompok.Kelompok murni (satu kelas) → Gini 0. Campuran 50:50 → Gini 0.5.

In [ ]:
def gini_impurity(labels):
    """Gini impurity dari array label (mis. 0/1). Kelompok kosong -> 0.0."""
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# print(gini_impurity([0, 0, 1, 1]))   # 0.5
# print(gini_impurity([0, 0, 0, 0]))   # 0.0
# print(gini_impurity([0, 1, 1]))      # 1 - (1/9 + 4/9) = 4/9


In [ ]:
# ✅ Cek latihan 5.1
assert abs(gini_impurity([0, 0, 1, 1]) - 0.5) < 1e-12
assert gini_impurity([0, 0, 0, 0]) == 0.0, 'murni -> 0'
assert abs(gini_impurity([0, 1, 1]) - 4 / 9) < 1e-12
assert gini_impurity([]) == 0.0, 'kelompok kosong -> 0.0'
assert abs(gini_impurity([1, 1, 1, 0]) - 2 * 0.25 * 0.75) < 1e-12
print('✅ Latihan 5.1 benar!')


In [ ]:
class MiniTree:
    """Decision tree klasifikasi mini: split Gini, max_depth terbatas.
    Cukup untuk MEMAHAMI mekanisme — bukan pengganti sklearn di produksi.
    """

    def __init__(self, max_depth=3, min_samples=2):
        self.max_depth = max_depth
        self.min_samples = min_samples

    def _best_split(self, X, y):
        m, d = X.shape
        best = (np.inf, None, None)          # (gini tertimbang, fitur, threshold)
        for f in range(d):
            for t in np.unique(X[:, f])[:-1]:
                mask = X[:, f] <= t
                if mask.sum() < self.min_samples or (~mask).sum() < self.min_samples:
                    continue
                g = (mask.mean() * gini_impurity(y[mask])
                     + (1 - mask.mean()) * gini_impurity(y[~mask]))
                if g < best[0]:
                    best = (g, f, t)
        return best[1], best[2]

    def fit(self, X, y, depth=0):
        # Leaf: cukup dalam / murni / tak ada split valid
        if depth >= self.max_depth or gini_impurity(y) == 0.0 or len(y) < 2 * self.min_samples:
            self.prediksi = float(np.mean(y) >= 0.5)
            return self
        f, t = self._best_split(X, y)
        if f is None:
            self.prediksi = float(np.mean(y) >= 0.5)
            return self
        mask = X[:, f] <= t
        self.split = (f, t)
        self.kiri = MiniTree(self.max_depth, self.min_samples).fit(X[mask], y[mask], depth + 1)
        self.kanan = MiniTree(self.max_depth, self.min_samples).fit(X[~mask], y[~mask], depth + 1)
        self.prediksi = None
        return self

    def predict_one(self, x):
        node = self
        while node.prediksi is None:
            f, t = node.split
            node = node.kiri if x[f] <= t else node.kanan
        return int(node.prediksi)

    def predict(self, X):
        return np.array([self.predict_one(x) for x in np.asarray(X)])

# Uji: dataset two-moons sederhana
rng = np.random.default_rng(3)
n_per = 100
Xa = np.column_stack([rng.normal(0, 0.6, n_per), rng.normal(0, 0.6, n_per)])
ya = np.zeros(n_per, dtype=int)
Xb = np.column_stack([rng.normal(2.2, 0.6, n_per), rng.normal(1.2, 0.6, n_per)])
yb = np.ones(n_per, dtype=int)
Xtree = np.vstack([Xa, Xb])
ytree = np.concatenate([ya, yb])

for depth in [1, 2, 5]:
    tree = MiniTree(max_depth=depth).fit(Xtree, ytree)
    acc = float((tree.predict(Xtree) == ytree).mean())
    print(f'max_depth={depth}: accuracy train = {acc:.2f}')


**Observasi:** depth 1 (stump) hanya bisa SATU garis potong — underfit.Depth 5 bebas melengkung — akurasi train tinggi. Pada data nyata, depth besartanpa pembatasan = overfitting (sklearn default tidak membatasi — hati-hati).

### 🎯 Latihan 5.2 — Random Forest = VotingForest: banyak tree, masing-masing dilatih di **bootstrap sample** (barisacak dengan pengembalian) dan melihat subset fitur acak. Prediksi akhir =**mayoritas**. Kekuatan kolektif mengalahkan satu tree yang rapuh.

In [ ]:
def forest_vote(preds):
    """preds: list of array 0/1 (hasil tiap tree, sama panjang).
    Return array mayoritas per posisi. Kalau seri -> pilih 1.
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# p = forest_vote([[1, 0, 1], [1, 1, 1], [0, 0, 1]])
# print(p)   # [1, 0, 1]


In [ ]:
# ✅ Cek latihan 5.2
assert forest_vote([[1, 0, 1], [1, 1, 1], [0, 0, 1]]).tolist() == [1, 0, 1]
assert forest_vote([[0], [1]]).tolist() == [1], 'seri -> 1'
assert forest_vote([[0, 0], [0, 1], [0, 1]]).tolist() == [0, 1]
print('✅ Latihan 5.2 benar!')


In [ ]:
# Demo forest mini: bootstrap + random feature subset + vote
rng = np.random.default_rng(11)
trees = []
n_tree = 25
for i in range(n_tree):
    idx = rng.choice(len(Xtree), size=len(Xtree), replace=True)   # bootstrap
    tree = MiniTree(max_depth=3).fit(Xtree[idx], ytree[idx])
    trees.append(tree)

all_preds = [t.predict(Xtree) for t in trees]
forest_pred = forest_vote(all_preds)
single_acc = float((trees[0].predict(Xtree) == ytree).mean())
forest_acc = float((forest_pred == ytree).mean())
print(f'accuracy 1 tree  : {single_acc:.3f}')
print(f'accuracy forest  : {forest_acc:.3f}  ({n_tree} tree, bootstrap + vote)')
print('-> Di data nyata random forest/XGBoost jauh lebih kuat — ini intuisinya.')


## Bagian 6 — K-Means dari NolUnsupervised: tidak ada label. K-means mengelompokkan titik berdasarkanjarak: assign ke centroid terdekat → update centroid = mean anggota → ulangi.

In [ ]:
rng = np.random.default_rng(5)
blob1 = rng.normal([0, 0], 0.5, size=(80, 2))
blob2 = rng.normal([4, 3], 0.5, size=(60, 2))
blob3 = rng.normal([-3, 4], 0.5, size=(50, 2))
Xk = np.vstack([blob1, blob2, blob3])

def kmeans_1run(X, k, seed=0, n_iter=50):
    """Satu jalanan k-means. Return (labels, centroids)."""
    r = np.random.default_rng(seed)
    cent = X[r.choice(len(X), size=k, replace=False)].copy()
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - cent[None]) ** 2).sum(axis=2)
        lab = d2.argmin(axis=1)
        new = np.array([X[lab == j].mean(axis=0) if (lab == j).any() else cent[j]
                        for j in range(k)])
        if np.allclose(new, cent):
            break
        cent = new
    return lab, cent

labels_k, cents = kmeans_1run(Xk, k=3, seed=0)

plt.figure(figsize=(6, 4.5))
plt.scatter(Xk[:, 0], Xk[:, 1], c=labels_k, cmap='viridis', alpha=0.6)
plt.scatter(cents[:, 0], cents[:, 1], c='red', s=200, marker='X', label='centroid')
plt.title('K-means menemukan 3 grup TANPA label'); plt.legend(); plt.show()

print('ukuran klaster:', np.bincount(labels_k))   # ~[80, 60, 50] (urutan bisa beda)


### 🎯 Latihan 6.1 — Assign ke Centroid TerdekatLangkah **assign** adalah jantung k-means (dan juga dasar algoritmanearest-neighbor, vector search di RAG Bab 12, dan kuantisasi embedding).

In [ ]:
def assign_labels(X, centroids):
    """Return label tiap titik: indeks centroid TERDEKAT (jarak kuadrat).
    X: (n, d), centroids: (k, d) -> labels: (n,) int.
    Tanpa loop per titik (hint: broadcasting + argmin).
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# c = np.array([[0.0, 0.0], [10.0, 10.0]])
# pts = np.array([[1., 1.], [9., 9.], [0.5, 0.]])
# print(assign_labels(pts, c))   # [0, 1, 0]


In [ ]:
# ✅ Cek latihan 6.1
c = np.array([[0.0, 0.0], [10.0, 10.0]])
pts = np.array([[1., 1.], [9., 9.], [0.5, 0.], [11., 10.]])
lab = assign_labels(pts, c)
assert lab.tolist() == [0, 1, 0, 1]
assert lab.dtype.kind in 'iu', 'label harus integer'
c3 = np.array([[0.0], [5.0], [100.0]])
lab3 = assign_labels(np.array([[4.9], [99.0], [0.1]]), c3)
assert lab3.tolist() == [1, 2, 0]
print('✅ Latihan 6.1 benar!')


In [ ]:
# Sanity check: assign_labels harus konsisten dengan kmeans_1run
lab_check = assign_labels(Xk, cents)
same = float((lab_check == labels_k).mean())
print(f'kesesuaian assign vs hasil kmeans: {same:.1%}')
assert same > 0.99, 'fungsimu harus mereproduksi assignment k-means'


## 🏁 Penutup Lab**Refleksi:**1. Jelaskan ke teman non-teknis: kenapa accuracy 90% bisa dari model yang useless?2. Kapan memilih recall tinggi vs precision tinggi? Beri 1 contoh bisnis masing-masing.3. Bagian mana yang paling "klik": melatih model (1–2), mengevaluasi (3),   overfitting (4), atau model berbasis pohon & clustering (5–6)?   Itu petunjuk bab lanjutan yang perlu kamu dalami.**Lanjut:** `02_kuis_ml_klasik.ipynb` → `project-starter-kampanye-marketing/` → `cheatsheet-ml-klasik.md`.